# 从 PokemonDB 抓取宝可梦可学会的招式数据（v2）

本 notebook 会读取 `../data/move.csv` 和 `../data/pokemon151_v3.csv`，从 `https://pokemondb.net/pokedex/[name_en]/moves/[generation]` 检查世代 9 到 1，仅抓取第一个存在学习招式数据的最新世代，导出为 `../data/learn_v2.csv`。

In [1]:
import csv
import re
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

BASE_URL = 'https://pokemondb.net'
GEN_CHECK_ORDER = list(range(9, 0, -1))
POKEMON_CSV = Path('../data/pokemon151_v3.csv').resolve()
MOVE_CSV = Path('../data/move.csv').resolve()
OUTPUT_CSV = Path('../data/learn_v2.csv').resolve()
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
}
RATE_LIMIT_SECONDS = 0.05
MAX_WORKERS = 5

session = requests.Session()
session.headers.update(HEADERS)
adapter = requests.adapters.HTTPAdapter(pool_connections=20, pool_maxsize=20)
session.mount('https://', adapter)

RE_LEVEL = re.compile(r'(\d+)')
RE_GEN = re.compile(r'generation\s*(\d+)', re.I)


def slugify_pokemon_name(name: str):
    slug = name.strip().lower()
    slug = slug.replace('♀', '-f').replace('♂', '-m')
    slug = slug.replace('.', '').replace("'", '').replace('é', 'e')
    slug = re.sub(r'[^a-z0-9-]+', '-', slug)
    slug = re.sub(r'-{2,}', '-', slug).strip('-')
    return slug


def get_soup(url: str):
    response = session.get(url, timeout=20)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    time.sleep(RATE_LIMIT_SECONDS)
    return soup


def load_csv(path: Path):
    with open(path, 'r', encoding='utf-8', newline='') as fp:
        return list(csv.DictReader(fp))


def build_move_lookup(moves):
    lookup = {}
    for row in moves:
        name = row.get('name_en', '').strip().lower()
        if not name:
            continue
        lookup[name] = row.get('move_id', '')
    return lookup


def normalize_method(title: str):
    text = (title or '').lower()
    if 'level up' in text or text == 'level up':
        return 'level_up'
    if 'egg' in text:
        return 'egg'
    if 'breeding' in text:
        return 'breeding'
    if 'hm' in text:
        return 'HM'
    if 'tm' in text and 'tr' in text:
        return 'TM'
    if 'tm' in text:
        return 'TM'
    if 'tr' in text:
        return 'TR'
    if 'tutor' in text:
        return 'tutor'
    if 'transfer-only' in text or 'transfer only' in text or 'transfer' in text:
        return 'transfer'
    if 'pre evolution' in text or 'pre-evolution' in text:
        return 'pre_evolution'
    if 'evolution' in text:
        return 'evolution'
    return 'transfer'


def parse_level(value: str):
    match = RE_LEVEL.search((value or '').strip())
    return match.group(1) if match else ''


def find_next_table(heading):
    table = heading.find_next('table')
    if table:
        return table
    div = heading.find_next('div')
    if div:
        return div.select_one('table.data-table')
    return None


def infer_method_transfer(text: str):
    text = (text or '').lower()
    if 'level' in text:
        return 'level_up'
    if 'egg' in text:
        return 'egg'
    if 'breed' in text:
        return 'breeding'
    if 'hm' in text:
        return 'HM'
    if 'tr' in text:
        return 'TR'
    if 'tm' in text:
        return 'TM'
    if 'tutor' in text:
        return 'tutor'
    if 'evol' in text:
        return 'evolution'
    return 'transfer'


def parse_transfer_gen(text: str, default_gen: str):
    if not text:
        return default_gen
    match = RE_GEN.search(text)
    if match:
        return match.group(1)
    return default_gen


def parse_learn_soup(pokemon, soup, gen, move_lookup):
    learn_rows = []
    latest_generation = str(gen)
    headings = soup.select('h2,h3,h4')
    if headings:
        for heading in headings:
            title = heading.get_text(strip=True)
            method = normalize_method(title)
            table = find_next_table(heading)
            if table is None:
                continue
            header_cells = [th.get_text(strip=True).lower() for th in table.select('thead th')]
            level_idx = None
            for i, h in enumerate(header_cells):
                if 'level' in h or h.replace('.', '').strip() in ('lv', 'lv'):
                    level_idx = i
                    break
            rows = table.select('tbody tr')
            prev_heading = title.lower()
            pokemon_name_lower = pokemon['name_en'].strip().lower()
            if pokemon_name_lower in prev_heading and prev_heading != pokemon_name_lower:
                continue
            for row in rows:
                cells = row.select('td')
                if not cells:
                    continue
                name_link = row.select_one('a.ent-name')
                if not name_link:
                    continue
                move_name_en = name_link.get_text(strip=True)
                move_id = move_lookup.get(move_name_en.strip().lower(), '')
                row_text = ' '.join(td.get_text(' ', strip=True) for td in cells)
                row_method = method
                method_transfer = ''
                learn_level = ''
                gen_value = latest_generation
                if method == 'transfer':
                    method_transfer = infer_method_transfer(f"{title} {row_text}")
                    gen_value = parse_transfer_gen(f"{title} {row_text}", latest_generation)
                if method in ('TM', 'TR') or (method == 'transfer' and header_cells and header_cells[0].lower().startswith(('tm', 'tr'))):
                    marker = cells[0].get_text(strip=True).lower()
                    if 'tm' in marker:
                        row_method = 'TM'
                    elif 'tr' in marker:
                        row_method = 'TR'
                    else:
                        row_method = method
                if method == 'level_up':
                    if level_idx is not None and level_idx < len(cells):
                        learn_level = parse_level(cells[level_idx].get_text(strip=True))
                    else:
                        learn_level = parse_level(cells[0].get_text(strip=True))
                elif method == 'egg':
                    learn_level = '1'
                elif method in ['breeding', 'HM', 'TM', 'TR', 'tutor']:
                    learn_level = '0'
                elif method == 'evolution':
                    learn_level = str(pokemon.get('evolution_level', '')).strip()
                elif method == 'pre_evolution':
                    learn_level = '0'
                elif method == 'transfer':
                    learn_level = '0'
                if method in ('evolution', 'pre_evolution') and not (learn_level or '').isdigit():
                    learn_level = '0'
                if method == 'transfer' and not method_transfer:
                    method_transfer = 'transfer'
                learn_rows.append({
                    'pokemon_id': pokemon['pokemon_id'],
                    'pokemon_name_en': pokemon['name_en'],
                    'move_name_en': move_name_en,
                    'move_id': move_id,
                    'gen': gen_value,
                    'method': row_method,
                    'method_transfer': method_transfer,
                    'learn_level': learn_level,
                })
    else:
        for table in soup.select('table.data-table'):
            prev_heading = table.find_previous(lambda tag: tag.name in ['h2', 'h3', 'h4'])
            title = prev_heading.get_text(strip=True) if prev_heading else ''
            method = normalize_method(title)
            header_cells = [th.get_text(strip=True).lower() for th in table.select('thead th')]
            level_idx = None
            for i, h in enumerate(header_cells):
                if 'level' in h or h.replace('.', '').strip() in ('lv', 'lv'):
                    level_idx = i
                    break
            prev_text = title.lower()
            pokemon_name_lower = pokemon['name_en'].strip().lower()
            if pokemon_name_lower in prev_text and prev_text != pokemon_name_lower:
                continue
            rows = table.select('tbody tr')
            for row in rows:
                cells = row.select('td')
                if not cells:
                    continue
                name_link = row.select_one('a.ent-name')
                if not name_link:
                    continue
                move_name_en = name_link.get_text(strip=True)
                move_id = move_lookup.get(move_name_en.strip().lower(), '')
                row_text = ' '.join(td.get_text(' ', strip=True) for td in cells)
                row_method = method
                method_transfer = ''
                learn_level = ''
                gen_value = latest_generation
                if method == 'transfer':
                    method_transfer = infer_method_transfer(f"{title} {row_text}")
                    gen_value = parse_transfer_gen(f"{title} {row_text}", latest_generation)
                if method in ('TM', 'TR') or (method == 'transfer' and header_cells and header_cells[0].lower().startswith(('tm', 'tr'))):
                    marker = cells[0].get_text(strip=True).lower()
                    if 'tm' in marker:
                        row_method = 'TM'
                    elif 'tr' in marker:
                        row_method = 'TR'
                    else:
                        row_method = method
                if method == 'level_up':
                    if level_idx is not None and level_idx < len(cells):
                        learn_level = parse_level(cells[level_idx].get_text(strip=True))
                    else:
                        learn_level = parse_level(cells[0].get_text(strip=True))
                elif method == 'egg':
                    learn_level = '1'
                elif method in ['breeding', 'HM', 'TM', 'TR', 'tutor']:
                    learn_level = '0'
                elif method == 'evolution':
                    learn_level = str(pokemon.get('evolution_level', '')).strip()
                elif method == 'pre_evolution':
                    learn_level = '0'
                elif method == 'transfer':
                    learn_level = '0'
                if method in ('evolution', 'pre_evolution') and not (learn_level or '').isdigit():
                    learn_level = '0'
                if method == 'transfer' and not method_transfer:
                    method_transfer = 'transfer'
                learn_rows.append({
                    'pokemon_id': pokemon['pokemon_id'],
                    'pokemon_name_en': pokemon['name_en'],
                    'move_name_en': move_name_en,
                    'move_id': move_id,
                    'gen': gen_value,
                    'method': row_method,
                    'method_transfer': method_transfer,
                    'learn_level': learn_level,
                })
    return learn_rows


def try_fetch_soup(slug: str, gen: int):
    url = f'{BASE_URL}/pokedex/{slug}/moves/{gen}'
    try:
        return get_soup(url), url
    except requests.HTTPError:
        alt_slugs = [slug]
        if slug.endswith('-f'):
            alt_slugs.append(slug.replace('-f', 'f'))
        if slug.endswith('-m'):
            alt_slugs.append(slug.replace('-m', 'm'))
        for alt_slug in alt_slugs[1:]:
            alt_url = f'{BASE_URL}/pokedex/{alt_slug}/moves/{gen}'
            try:
                return get_soup(alt_url), alt_url
            except requests.HTTPError:
                continue
    return None, url


def parse_learn_page(pokemon, move_lookup):
    slug = slugify_pokemon_name(pokemon['name_en'])
    # 从最新世代向下检查，找到第一代有学习数据的页面后立即返回结果
    for gen in GEN_CHECK_ORDER:
        soup, fetched_url = try_fetch_soup(slug, gen)
        if soup is None:
            continue
        learn_rows = parse_learn_soup(pokemon, soup, gen, move_lookup)
        if learn_rows:
            print(f'DEBUG: fetched {fetched_url} gen={gen} rows={len(learn_rows)}')
            return learn_rows
    print(f'Warning: 无法在 {slug} 的 9-1 代页面中找到学习招式，跳过 {pokemon["name_en"]}')
    return []


def save_csv(learn_data):
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        'learn_id',
        'pokemon_id',
        'move_id',
        'pokemon_name_en',
        'move_name_en',
        'gen',
        'method',
        'method_transfer',
        'learn_level',
    ]
    with open(OUTPUT_CSV, 'w', encoding='utf-8', newline='') as fp:
        writer = csv.DictWriter(fp, fieldnames=fieldnames)
        writer.writeheader()
        for row in learn_data:
            writer.writerow({key: row.get(key, '') for key in fieldnames})


def main():
    pokemon_rows = load_csv(POKEMON_CSV)
    move_rows = load_csv(MOVE_CSV)
    move_lookup = build_move_lookup(move_rows)
    all_learn = []
    learn_id = 1

    def fetch_pokemon(pokemon):
        return parse_learn_page(pokemon, move_lookup)

    with ThreadPoolExecutor(max_workers=min(MAX_WORKERS, len(pokemon_rows))) as executor:
        for pokemon, learn_entries in zip(pokemon_rows, executor.map(fetch_pokemon, pokemon_rows)):
            for entry in learn_entries:
                entry['learn_id'] = learn_id
                all_learn.append(entry)
                learn_id += 1

    save_csv(all_learn)
    print(f'完成：已保存 {len(all_learn)} 条学习招式数据到 {OUTPUT_CSV}')

main()

DEBUG: fetched https://pokemondb.net/pokedex/charmander/moves/9 gen=9 rows=109
DEBUG: fetched https://pokemondb.net/pokedex/venusaur/moves/9 gen=9 rows=153
DEBUG: fetched https://pokemondb.net/pokedex/bulbasaur/moves/9 gen=9 rows=89
DEBUG: fetched https://pokemondb.net/pokedex/ivysaur/moves/9 gen=9 rows=122
DEBUG: fetched https://pokemondb.net/pokedex/wartortle/moves/9 gen=9 rows=129
DEBUG: fetched https://pokemondb.net/pokedex/charizard/moves/9 gen=9 rows=145
DEBUG: fetched https://pokemondb.net/pokedex/squirtle/moves/9 gen=9 rows=102
DEBUG: fetched https://pokemondb.net/pokedex/blastoise/moves/9 gen=9 rows=178
DEBUG: fetched https://pokemondb.net/pokedex/charmeleon/moves/9 gen=9 rows=107
DEBUG: fetched https://pokemondb.net/pokedex/butterfree/moves/8 gen=8 rows=131
DEBUG: fetched https://pokemondb.net/pokedex/caterpie/moves/8 gen=8 rows=11
DEBUG: fetched https://pokemondb.net/pokedex/weedle/moves/9 gen=9 rows=10
DEBUG: fetched https://pokemondb.net/pokedex/kakuna/moves/9 gen=9 rows=1